In [ ]:
numeric_features = data.select_dtypes(include=['int', 'float']).columns.tolist() # get numeric feature names
print("Numeric features:", numeric_features) # print numeric feature names

text_features = data.select_dtypes(include="string").columns.tolist() # get text feature names
print("Text features:", text_features) # print text feature names

Numeric features: ['anio_inicio', 'anio_hecho', 'latitud', 'longitud']
Text features: ['mes_inicio', 'fecha_inicio', 'hora_inicio', 'mes_hecho', 'fecha_hecho', 'hora_hecho', 'delito', 'categoria_delito', 'competencia', 'fiscalia', 'agencia', 'unidad_investigacion', 'colonia_hecho', 'colonia_catalogo', 'alcaldia_hecho', 'alcaldia_catalogo', 'municipio_hecho']


In [ ]:
# Describe the statistics of the data (only numeric columns)
data.describe()

,anio_inicio,anio_hecho,latitud,longitud
count,"2,098,743.00","2,098,184.00","1,997,536.00","1,997,536.00"
mean,"2,020.10","2,019.94",19.39,-99.14
std,2.58,3.07,0.07,0.06
min,"2,016.00",222.00,19.10,-100.23
25%,"2,018.00","2,018.00",19.34,-99.18
50%,"2,020.00","2,020.00",19.39,-99.14
75%,"2,022.00","2,022.00",19.44,-99.10
max,"2,025.00","2,025.00",19.58,-98.95


In [ ]:
data[text_features].describe() # describe only the string columns

,mes_inicio,fecha_inicio,hora_inicio,mes_hecho,fecha_hecho,hora_hecho,delito,categoria_delito,competencia,fiscalia,agencia,unidad_investigacion,colonia_hecho,colonia_catalogo,alcaldia_hecho,alcaldia_catalogo,municipio_hecho
count,2098743,2098740,2098728,2098184,2098183,2097856,2098743,2098743,1034725,2098741,2098743,2097765,1996619,1974303,2073847,17586,2098743
unique,19,56540,115107,19,32513,2872,357,18,3,100,230,158,1699,1423,17,1,1
top,Octubre,2020-03-06,00:00:00,Enero,2021-01-01,12:00:00,VIOLENCIA FAMILIAR,DELITO DE BAJO IMPACTO,FUERO COMUN,AGENCIA DE DENUNCIA DIGITAL,CEN-1,UI-1SD,CENTRO,Centro,CUAUHTEMOC,CDMX (indeterminada),CDMX
freq,185141,934,1484811,183889,1259,192234,261181,1711755,1000272,150858,72170,737376,65086,65266,318787,17586,2098743


# AQUI EMPIEZA ADOLF FASE 1

In [ ]:
# ============================================================================
# PASO 4: Asignación por similitud TF-IDF para no asignados
# ============================================================================
if no_asignados > 0:
    print(f"\n{'='*80}")
    print(f"ASIGNACIÓN POR SIMILITUD TF-IDF (casos no cubiertos por reglas):")
    print(f"{'='*80}")
    
    # Crear corpus: textos ya asignados agrupados por macro
    asignados_df = df_mapeo[df_mapeo['macro_categoria'].notna()]
    
    # Centroide textual por macro: concatenar todos los nombres de esa macro
    centroides_texto = {}
    for macro in asignados_df['macro_categoria'].unique():
        textos = asignados_df[asignados_df['macro_categoria'] == macro]['delito_normalizado'].tolist()
        centroides_texto[macro] = ' '.join(textos)
    
    # Vectorizar con TF-IDF char n-grams
    macros_lista = list(centroides_texto.keys())
    textos_centroides = [centroides_texto[m] for m in macros_lista]
    
    no_asig_textos = no_asig['delito_normalizado'].tolist()
    
    todos_textos = textos_centroides + no_asig_textos
    
    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=5000
    )
    tfidf_matrix = vectorizer.fit_transform(todos_textos)
    
    # Separar matrices
    centroides_vec = tfidf_matrix[:len(macros_lista)]
    no_asig_vec = tfidf_matrix[len(macros_lista):]
    
    # Calcular similitud
    sims = cosine_similarity(no_asig_vec, centroides_vec)
    
    for i, (_, row) in enumerate(no_asig.iterrows()):
        mejor_idx = sims[i].argmax()
        mejor_sim = sims[i][mejor_idx]
        macro_asignada = macros_lista[mejor_idx]
        
        # Umbral de confianza
        confianza = 'ALTA' if mejor_sim > 0.3 else 'BAJA'
        
        print(f"\n  '{row['delito_original']}' ({row['conteo']:,} registros)")
        print(f"    → {macro_asignada} (similitud: {mejor_sim:.3f}, confianza: {confianza})")
        
        # Top 3 candidatos
        top3 = sims[i].argsort()[-3:][::-1]
        print(f"    Top 3: {', '.join([f'{macros_lista[j]} ({sims[i][j]:.3f})' for j in top3])}")
        
        # Asignar
        idx_original = df_mapeo[df_mapeo['delito_original'] == row['delito_original']].index[0]
        df_mapeo.loc[idx_original, 'macro_categoria'] = macro_asignada
        df_mapeo.loc[idx_original, 'metodo_asignacion'] = f'tfidf_sim={mejor_sim:.3f}'

# ============================================================================
# PASO 5: Generar nombre limpio (delito_limpio)
# ============================================================================
def limpiar_nombre_delito(texto):
    """Genera versión limpia del nombre del delito."""
    t = normalizar_texto(texto)
    # Remover especificaciones muy largas entre paréntesis
    # pero mantener las informativas cortas
    partes = re.findall(r'\(([^)]+)\)', t)
    for parte in partes:
        if len(parte) > 50:  # Paréntesis muy largos → eliminar
            t = t.replace(f'({parte})', '')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df_mapeo['delito_limpio'] = df_mapeo['delito_original'].apply(limpiar_nombre_delito)

# Consolidar duplicados por normalización: usar el nombre más frecuente
for norm, grupo in df_mapeo.groupby('delito_normalizado'):
    if len(grupo) > 1:
        # El delito_limpio es el mismo para todos (post normalización)
        nombre_canonico = grupo.sort_values('conteo', ascending=False).iloc[0]['delito_limpio']
        df_mapeo.loc[grupo.index, 'delito_limpio'] = nombre_canonico

# ============================================================================
# PASO 6: Reporte final de la taxonomía
# ============================================================================
print(f"\n{'='*80}")
print(f"TAXONOMÍA CANÓNICA — RESUMEN")
print(f"{'='*80}")

for macro in df_mapeo.groupby('macro_categoria')['conteo'].sum().sort_values(ascending=False).index:
    grupo = df_mapeo[df_mapeo['macro_categoria'] == macro]
    total = grupo['conteo'].sum()
    pct = total / df_mapeo['conteo'].sum() * 100
    n_cats = len(grupo)
    print(f"\n▸ {macro} — {total:,} registros ({pct:.1f}%) — {n_cats} categorías originales")
    
    # Top 5 delitos dentro de cada macro
    top = grupo.sort_values('conteo', ascending=False).head(5)
    for _, row in top.iterrows():
        metodo = row['metodo_asignacion']
        flag = ' ⚠️' if 'tfidf' in str(metodo) and 'BAJA' in str(metodo) else ''
        print(f"    {row['conteo']:>8,}  {row['delito_limpio']}{flag}")
    if len(grupo) > 5:
        print(f"    ... y {len(grupo)-5} más")

# ============================================================================
# PASO 7: Exportar mapeo y dataset aumentado
# ============================================================================
# 7a. CSV de mapeo
mapeo_export = df_mapeo[['delito_original', 'delito_limpio', 'macro_categoria', 
                          'metodo_asignacion', 'conteo']].copy()
mapeo_export = mapeo_export.sort_values(['macro_categoria', 'conteo'], ascending=[True, False])
mapeo_export.to_csv('taxonomia_delitos_mapeo.csv', index=False, encoding='utf-8-sig')
print(f"\n✓ Mapeo exportado: taxonomia_delitos_mapeo.csv ({len(mapeo_export)} filas)")

# 7b. Aplicar al dataset completo
dict_macro = dict(zip(df_mapeo['delito_original'], df_mapeo['macro_categoria']))
dict_limpio = dict(zip(df_mapeo['delito_original'], df_mapeo['delito_limpio']))

df['macro_categoria'] = df['delito'].map(dict_macro)
df['delito_limpio'] = df['delito'].map(dict_limpio)

# Verificar que no quedaron nulos
nulos_macro = df['macro_categoria'].isna().sum()
nulos_limpio = df['delito_limpio'].isna().sum()
print(f"  Nulos en macro_categoria: {nulos_macro}")
print(f"  Nulos en delito_limpio: {nulos_limpio}")

if nulos_macro > 0:
    print(f" Delitos sin macro asignada:")
    for d in df[df['macro_categoria'].isna()]['delito'].unique():
        print(f"      {d}")

# 7c. Exportar dataset con taxonomía
df.to_csv('carpetasFGJ_fase2.csv', index=False, encoding='utf-8-sig')
print(f"✓ Dataset exportado: carpetasFGJ_fase2.csv ({len(df):,} registros, {df.shape[1]} columnas)")

# ============================================================================
# PASO 8: Estadísticas para validación
# ============================================================================
print(f"\n{'='*80}")
print(f"ESTADÍSTICAS DE VALIDACIÓN")
print(f"{'='*80}")

print(f"\n  Categorías originales:      {df['delito'].nunique()}")
print(f"  Categorías limpias:         {df['delito_limpio'].nunique()}")
print(f"  Macro-categorías:           {df['macro_categoria'].nunique()}")
print(f"  Reducción efectiva:         {df['delito'].nunique()} → {df['delito_limpio'].nunique()} → {df['macro_categoria'].nunique()}")

print(f"\n  Distribución por macro-categoría:")
dist = df['macro_categoria'].value_counts()
for macro, count in dist.items():
    pct = count/len(df)*100
    bar = '█' * int(pct)
    print(f"    {macro:40s} {count:>9,} ({pct:>5.1f}%) {bar}")

print(f"\n{'='*80}")
print(f"SIGUIENTE PASO: Revisa taxonomia_delitos_mapeo.csv")
print(f"Valida especialmente las asignaciones por TF-IDF (método ≠ 'regla').")
print(f"Una vez validado, carpetasFGJ_fase2.csv es tu input para la etapa 5.2.")
print(f"{'='*80}")

# FASE 3 ADOLF

In [ ]:
"""
=============================================================================
Urban Crime CDMX — Etapa 5.3: Aprendizaje de representaciones urbanas
=============================================================================
Input:  firmas_h3.csv (1,061 hexágonos × 45 dims)
        h3_metadata.csv (metadatos por hexágono)

Métodos:
  A) Línea base — PCA, NMF, K-Means, GMM, HDBSCAN
  B) Autoencoder denso
  C) Comparación de representaciones

Output: embeddings_h3.csv — embeddings finales por hexágono
        clusters_h3.csv — asignación de clusters por método
        modelo_autoencoder.pth — pesos del autoencoder
=============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA, NMF
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# PASO 0: Cargar datos
# ============================================================================
print("Cargando firmas...")
firmas = pd.read_csv("firmas_h3.csv", index_col='h3_id')
metadata = pd.read_csv("h3_metadata.csv", index_col='h3_id')

print(f"Hexágonos: {len(firmas):,}")
print(f"Dimensiones: {firmas.shape[1]}")

# ============================================================================
# PASO 1: Preprocesamiento
# ============================================================================
# Separar features proporcionales de features absolutas
cols_proporciones = [c for c in firmas.columns if c not in ['intensidad_log', 'ratio_violencia']]
cols_extra = ['intensidad_log', 'ratio_violencia']

# Para PCA y Autoencoder: estandarizar todo
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(firmas)

# Para NMF: usar MinMax (NMF requiere valores no negativos)
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(firmas)

print(f"Datos estandarizados: {X_std.shape}")

# ============================================================================
# PASO 2A: PCA
# ============================================================================
print(f"\n{'='*80}")
print(f"MÉTODO A1: PCA")
print(f"{'='*80}")

# PCA completo para ver varianza explicada
pca_full = PCA().fit(X_std)
varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)

# ¿Cuántas componentes para 80%, 90%, 95%?
for umbral in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(varianza_acum >= umbral) + 1
    print(f"  Componentes para {umbral*100:.0f}% varianza: {n_comp}")

# Usar componentes que expliquen 90% de la varianza
N_COMPONENTS_PCA = np.argmax(varianza_acum >= 0.90) + 1
pca = PCA(n_components=N_COMPONENTS_PCA)
X_pca = pca.fit_transform(X_std)

print(f"\n  PCA con {N_COMPONENTS_PCA} componentes:")
print(f"  Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Top features por componente (primeras 3)
feature_names = firmas.columns.tolist()
print(f"\n  Top features por componente:")
for i in range(min(3, N_COMPONENTS_PCA)):
    loadings = pd.Series(pca.components_[i], index=feature_names)
    top_pos = loadings.nlargest(3)
    top_neg = loadings.nsmallest(3)
    print(f"\n  PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}% varianza):")
    print(f"    (+) {', '.join([f'{n}: {v:.3f}' for n, v in top_pos.items()])}")
    print(f"    (-) {', '.join([f'{n}: {v:.3f}' for n, v in top_neg.items()])}")